<a href="https://colab.research.google.com/github/heetaamin/Assignment-4/blob/main/01_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# load the benchmark table built in notebook 00
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import time
from scipy import stats
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.combine import SMOTETomek

BASE = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 4/'
DATA_DIR = BASE + 'data/'
RESULTS_DIR = BASE + 'results/'
SEED = 42

vec = pd.read_csv(DATA_DIR + 'networkTraffic_vectors.csv')
dev = vec[vec['fold'] >= 0].reset_index(drop=True)

print("All vectors:", vec.shape)
print("Development set:", dev.shape)
print(dev['fold'].value_counts().sort_index())

Mounted at /content/drive
All vectors: (153684, 47)
Development set: (122947, 47)
fold
0    24590
1    24590
2    24589
3    24589
4    24589
Name: count, dtype: int64


In [ ]:
# per-fold preparation for the tree models, same steps as the A2 tree pipeline
# everything that depends on the data (proto threshold, dummy columns) is worked out
# on the training part only and then applied to the validation part
# no scaling - trees split one feature at a time so the ranges do not matter

META_COLS = ['label_first', 'label_majority', 'is_conflict', 'n_rows', 'fold']
TREE_DROP = ['ct_ftp_cmd', 'sloss', 'dloss', 'tcprtt']
NOMINAL_COLS = ['proto', 'state', 'service']

def prep_tree_fold(train, val, drop_cols=TREE_DROP):
    X_train = train.drop(columns=META_COLS + drop_cols).copy()
    X_val = val.drop(columns=META_COLS + drop_cols).copy()

    # group proto categories below 1% of the training rows into 'other'
    proto_counts = X_train['proto'].value_counts()
    keep_protos = proto_counts[proto_counts >= 0.01 * len(X_train)].index.tolist()
    X_train['proto'] = X_train['proto'].where(X_train['proto'].isin(keep_protos), 'other')
    X_val['proto'] = X_val['proto'].where(X_val['proto'].isin(keep_protos), 'other')

    # sklearn trees only take numeric input so the nominal features are one-hot encoded
    # '?' in service stays as its own category, same as A2
    X_train = pd.get_dummies(X_train, columns=NOMINAL_COLS)
    X_val = pd.get_dummies(X_val, columns=NOMINAL_COLS)
    X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

    return X_train, X_val, keep_protos

# quick check on one fold
train_check, val_check = dev[dev['fold'] != 0], dev[dev['fold'] == 0]
X_tr, X_va, kept = prep_tree_fold(train_check, val_check)
print("Train:", X_tr.shape, " Validation:", X_va.shape)
print("Kept protos:", kept)

Train: (98357, 59)  Validation: (24590, 59)
Kept protos: ['tcp', 'udp']


In [ ]:
# conflict treatment test
# same A2 tree configuration for all three (entropy, ccp_alpha=0.000129, random oversampling 5x)
# only the treatment of the 2445 conflicting vectors changes:
#   first    - keep the label of the first occurrence
#   majority - use the most common label for that vector
#   remove   - drop the conflicting vectors completely
# development folds only, the test set is not used for this decision

def oversample_5x(X, y):
    # each minority class is raised to 5x its size, capped at the majority class size
    counts = y.value_counts()
    target = {cls: min(n * 5, counts.max()) for cls, n in counts.items()}
    return RandomOverSampler(sampling_strategy=target, random_state=SEED).fit_resample(X, y)

treatment_results = []
for treatment in ['first', 'majority', 'remove']:
    data = dev[dev['is_conflict'] == 0] if treatment == 'remove' else dev
    label_col = 'label_majority' if treatment == 'majority' else 'label_first'

    for fold in range(5):
        train, val = data[data['fold'] != fold], data[data['fold'] == fold]
        X_train, X_val, _ = prep_tree_fold(train, val)
        X_res, y_res = oversample_5x(X_train, train[label_col])

        model = DecisionTreeClassifier(criterion='entropy', ccp_alpha=0.000129, random_state=SEED)
        model.fit(X_res, y_res)
        macro_f1 = f1_score(val[label_col], model.predict(X_val), average='macro')
        treatment_results.append({'treatment': treatment, 'fold': fold,
                                  'n_val': len(val), 'macro_f1': macro_f1})
    print(f"{treatment} done")

treatment_df = pd.DataFrame(treatment_results)
treatment_df.to_csv(RESULTS_DIR + 'conflict_treatment.csv', index=False)

print()
print(treatment_df.pivot(index='fold', columns='treatment', values='macro_f1').round(4))
print()
print(treatment_df.groupby('treatment')['macro_f1'].agg(['mean', 'std']).round(4))

first done
majority done
remove done

treatment   first  majority  remove
fold                               
0          0.6684    0.6841  0.6676
1          0.6516    0.6587  0.6502
2          0.6668    0.6579  0.6594
3          0.6586    0.6674  0.6632
4          0.6718    0.6922  0.6727

             mean     std
treatment                
first      0.6634  0.0082
majority   0.6720  0.0154
remove     0.6626  0.0085


In [ ]:
# revised single tree - re-tuning ccp_alpha on all 5 development folds
# in A2 this value was picked on one fold that was then also used for evaluation
# everything else is carried over from A2: entropy, random oversampling 5x, all 7 connection-count features
# majority label is used from here on

LABEL = 'label_majority'

# build the 5 folds once so they can be reused for the rest of the tree work
tree_folds = []
for fold in range(5):
    train, val = dev[dev['fold'] != fold], dev[dev['fold'] == fold]
    X_train, X_val, _ = prep_tree_fold(train, val)
    X_res, y_res = oversample_5x(X_train, train[LABEL])
    tree_folds.append((X_res, y_res, X_val, val[LABEL]))

alphas = [0, 0.00003, 0.00005, 0.00007, 0.0001, 0.000129, 0.00015, 0.0002, 0.0003, 0.0005, 0.001]

alpha_results = []
for alpha in alphas:
    for fold, (X_res, y_res, X_val, y_val) in enumerate(tree_folds):
        model = DecisionTreeClassifier(criterion='entropy', ccp_alpha=alpha, random_state=SEED)
        model.fit(X_res, y_res)
        alpha_results.append({'ccp_alpha': alpha, 'fold': fold,
                              'macro_f1': f1_score(y_val, model.predict(X_val), average='macro'),
                              'n_leaves': model.get_n_leaves()})
    print(f"ccp_alpha={alpha:.6f} done")

alpha_df = pd.DataFrame(alpha_results)
alpha_df.to_csv(RESULTS_DIR + 'tree_ccp_alpha.csv', index=False)

alpha_summary = alpha_df.groupby('ccp_alpha').agg(mean_f1=('macro_f1', 'mean'),
                                                  std_f1=('macro_f1', 'std'),
                                                  mean_leaves=('n_leaves', 'mean')).round(4)
print()
print(alpha_summary)

ccp_alpha=0.000000 done
ccp_alpha=0.000030 done
ccp_alpha=0.000050 done
ccp_alpha=0.000070 done
ccp_alpha=0.000100 done
ccp_alpha=0.000129 done
ccp_alpha=0.000150 done
ccp_alpha=0.000200 done
ccp_alpha=0.000300 done
ccp_alpha=0.000500 done
ccp_alpha=0.001000 done

           mean_f1  std_f1  mean_leaves
ccp_alpha                              
0.000000    0.6479  0.0167      10124.2
0.000030    0.6491  0.0172       6539.2
0.000050    0.6482  0.0123       4167.0
0.000070    0.6550  0.0128       2932.6
0.000100    0.6614  0.0156       1863.0
0.000129    0.6720  0.0154       1309.4
0.000150    0.6707  0.0113       1039.2
0.000200    0.6723  0.0132        677.8
0.000300    0.6695  0.0110        399.2
0.000500    0.6575  0.0119        228.0
0.001000    0.6039  0.0191        123.2


In [ ]:
# choose ccp_alpha with the one-standard-error rule (Breiman et al. 1984)
# the top of the table is flat, so instead of taking the single highest mean
# take the most heavily pruned tree that is still within one standard error of the best

best_alpha = alpha_summary['mean_f1'].idxmax()
best_mean = alpha_summary.loc[best_alpha, 'mean_f1']
std_err = alpha_summary.loc[best_alpha, 'std_f1'] / np.sqrt(5)
threshold = best_mean - std_err

TREE_ALPHA = alpha_summary[alpha_summary['mean_f1'] >= threshold].index.max()
print(f"Best mean: {best_mean:.4f} at ccp_alpha={best_alpha}")
print(f"One standard error: {std_err:.4f}, threshold: {threshold:.4f}")
print(f"Chosen ccp_alpha: {TREE_ALPHA}")

# final run of the revised tree on the development folds, keeping all the measures and the predictions
tree_results, tree_preds = [], []
for fold, (X_res, y_res, X_val, y_val) in enumerate(tree_folds):
    model = DecisionTreeClassifier(criterion='entropy', ccp_alpha=TREE_ALPHA, random_state=SEED)
    start = time.time()
    model.fit(X_res, y_res)
    fit_time = time.time() - start
    start = time.time()
    y_pred = model.predict(X_val)
    predict_time = time.time() - start

    tree_results.append({'model': 'tree', 'fold': fold,
                         'macro_f1': f1_score(y_val, y_pred, average='macro'),
                         'accuracy': accuracy_score(y_val, y_pred),
                         'weighted_f1': f1_score(y_val, y_pred, average='weighted'),
                         'balanced_accuracy': balanced_accuracy_score(y_val, y_pred),
                         'fit_time': fit_time, 'predict_time': predict_time})
    tree_preds.append(pd.DataFrame({'fold': fold, 'y_true': y_val.values, 'y_pred': y_pred}))

tree_results_df = pd.DataFrame(tree_results)
tree_results_df.to_csv(RESULTS_DIR + 'baseline_tree_cv.csv', index=False)
pd.concat(tree_preds).to_csv(RESULTS_DIR + 'baseline_tree_cv_preds.csv', index=False)

print()
print(tree_results_df.round(4))
print()
print(tree_results_df.drop(columns=['model', 'fold']).agg(['mean', 'std']).round(4))

Best mean: 0.6723 at ccp_alpha=0.0002
One standard error: 0.0059, threshold: 0.6664
Chosen ccp_alpha: 0.0003

  model  fold  macro_f1  accuracy  weighted_f1  balanced_accuracy  fit_time  \
0  tree     0    0.6821    0.7978       0.8122             0.7149    8.8542   
1  tree     1    0.6563    0.8029       0.8166             0.6869   10.4921   
2  tree     2    0.6612    0.8013       0.8143             0.7024   10.4011   
3  tree     3    0.6696    0.7986       0.8130             0.6998    8.8281   
4  tree     4    0.6784    0.8054       0.8180             0.7217   10.6494   

   predict_time  
0        0.0122  
1        0.0080  
2        0.0079  
3        0.0125  
4        0.0080  

      macro_f1  accuracy  weighted_f1  balanced_accuracy  fit_time  \
mean    0.6695    0.8012       0.8148             0.7051    9.8450   
std     0.0110    0.0031       0.0024             0.0136    0.9207   

      predict_time  
mean        0.0097  
std         0.0024  


In [ ]:
# checking the settings carried over from A2 still hold on the new data
# these were also picked on a single fold in A2, so each one is re-checked on all 5 folds
# ccp_alpha is kept at the chosen value, except for gini which needs its own values:
# gini and entropy are on different scales, so the same ccp_alpha prunes a gini tree much harder

def oversample(X, y, mult):
    if mult == 1:
        return X, y
    counts = y.value_counts()
    target = {cls: min(n * mult, counts.max()) for cls, n in counts.items()}
    return RandomOverSampler(sampling_strategy=target, random_state=SEED).fit_resample(X, y)

def tree_cv(criterion='entropy', alpha=TREE_ALPHA, mult=5, drop_cols=TREE_DROP):
    scores = []
    for fold in range(5):
        train, val = dev[dev['fold'] != fold], dev[dev['fold'] == fold]
        X_train, X_val, _ = prep_tree_fold(train, val, drop_cols)
        X_res, y_res = oversample(X_train, train[LABEL], mult)
        model = DecisionTreeClassifier(criterion=criterion, ccp_alpha=alpha, random_state=SEED)
        model.fit(X_res, y_res)
        scores.append(f1_score(val[LABEL], model.predict(X_val), average='macro'))
    return scores

checks = {
    'current tree (entropy, 5x, 4 features dropped)': tree_cv(),
    'no oversampling': tree_cv(mult=1),
    'oversampling 2x': tree_cv(mult=2),
    'oversampling 10x': tree_cv(mult=10),
    'keep the 4 redundant features': tree_cv(drop_cols=[]),
    'gini, ccp_alpha=0.00005': tree_cv(criterion='gini', alpha=0.00005),
    'gini, ccp_alpha=0.00007': tree_cv(criterion='gini', alpha=0.00007),
    'gini, ccp_alpha=0.0001': tree_cv(criterion='gini', alpha=0.0001),
}

check_df = pd.DataFrame(checks).T
check_df.columns = [f'fold_{i}' for i in range(5)]
check_df['mean'] = check_df.mean(axis=1)
check_df.to_csv(RESULTS_DIR + 'tree_setting_checks.csv')
print(check_df.round(4))

                                                fold_0  fold_1  fold_2  \
current tree (entropy, 5x, 4 features dropped)  0.6821  0.6563  0.6612   
no oversampling                                 0.6427  0.6323  0.6318   
oversampling 2x                                 0.6664  0.6415  0.6623   
oversampling 10x                                0.6851  0.6370  0.6514   
keep the 4 redundant features                   0.6833  0.6531  0.6614   
gini, ccp_alpha=0.00005                         0.6842  0.6556  0.6471   
gini, ccp_alpha=0.00007                         0.6910  0.6680  0.6672   
gini, ccp_alpha=0.0001                          0.6853  0.6478  0.6551   

                                                fold_3  fold_4    mean  
current tree (entropy, 5x, 4 features dropped)  0.6696  0.6784  0.6695  
no oversampling                                 0.6164  0.6407  0.6328  
oversampling 2x                                 0.6404  0.6481  0.6518  
oversampling 10x                         

In [ ]:
# checking the settings carried over from A2 still hold on the new data
# these were also picked on a single fold in A2, so each one is re-checked on all 5 folds
# ccp_alpha is kept at the chosen value, except for gini which needs its own values:
# gini and entropy are on different scales, so the same ccp_alpha prunes a gini tree much harder

def oversample(X, y, mult):
    if mult == 1:
        return X, y
    counts = y.value_counts()
    target = {cls: min(n * mult, counts.max()) for cls, n in counts.items()}
    return RandomOverSampler(sampling_strategy=target, random_state=SEED).fit_resample(X, y)

def tree_cv(criterion='entropy', alpha=TREE_ALPHA, mult=5, drop_cols=TREE_DROP):
    scores = []
    for fold in range(5):
        train, val = dev[dev['fold'] != fold], dev[dev['fold'] == fold]
        X_train, X_val, _ = prep_tree_fold(train, val, drop_cols)
        X_res, y_res = oversample(X_train, train[LABEL], mult)
        model = DecisionTreeClassifier(criterion=criterion, ccp_alpha=alpha, random_state=SEED)
        model.fit(X_res, y_res)
        scores.append(f1_score(val[LABEL], model.predict(X_val), average='macro'))
    return scores

checks = {
    'current tree (entropy, 5x, 4 features dropped)': tree_cv(),
    'no oversampling': tree_cv(mult=1),
    'oversampling 2x': tree_cv(mult=2),
    'oversampling 10x': tree_cv(mult=10),
    'keep the 4 redundant features': tree_cv(drop_cols=[]),
    'gini, ccp_alpha=0.00005': tree_cv(criterion='gini', alpha=0.00005),
    'gini, ccp_alpha=0.00007': tree_cv(criterion='gini', alpha=0.00007),
    'gini, ccp_alpha=0.0001': tree_cv(criterion='gini', alpha=0.0001),
}

check_df = pd.DataFrame(checks).T
check_df.columns = [f'fold_{i}' for i in range(5)]
check_df['mean'] = check_df.mean(axis=1)
check_df.to_csv(RESULTS_DIR + 'tree_setting_checks.csv')
print(check_df.round(4))

                                                fold_0  fold_1  fold_2  \
current tree (entropy, 5x, 4 features dropped)  0.6821  0.6563  0.6612   
no oversampling                                 0.6427  0.6323  0.6318   
oversampling 2x                                 0.6664  0.6415  0.6623   
oversampling 10x                                0.6851  0.6370  0.6514   
keep the 4 redundant features                   0.6833  0.6531  0.6614   
gini, ccp_alpha=0.00005                         0.6842  0.6556  0.6471   
gini, ccp_alpha=0.00007                         0.6910  0.6680  0.6672   
gini, ccp_alpha=0.0001                          0.6853  0.6478  0.6551   

                                                fold_3  fold_4    mean  
current tree (entropy, 5x, 4 features dropped)  0.6696  0.6784  0.6695  
no oversampling                                 0.6164  0.6407  0.6328  
oversampling 2x                                 0.6404  0.6481  0.6518  
oversampling 10x                         

In [ ]:
# kNN baseline - per-fold preparation, same steps as the A2 kNN pipeline
# kNN is distance based so it needs more preparation than the tree:
# redundant features removed, min-max scaling, and the top 10 features by mutual information
# everything is fitted on the training part only

CLUSTER_DROP = ['ct_dst_src_ltm', 'ct_src_dport_ltm', 'ct_dst_ltm', 'ct_src_ltm', 'ct_dst_sport_ltm']
KNN_DROP = TREE_DROP + CLUSTER_DROP

def prep_knn_fold(train, val, n_features=10):
    X_train = train.drop(columns=META_COLS + KNN_DROP).copy()
    X_val = val.drop(columns=META_COLS + KNN_DROP).copy()

    # flag for the missing service values, '?' is also kept as its own category
    X_train['service_missing'] = (X_train['service'] == '?').astype(int)
    X_val['service_missing'] = (X_val['service'] == '?').astype(int)

    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    scaler = MinMaxScaler()
    X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_val[numeric_cols] = scaler.transform(X_val[numeric_cols])

    proto_counts = X_train['proto'].value_counts()
    keep_protos = proto_counts[proto_counts >= 0.01 * len(X_train)].index.tolist()
    X_train['proto'] = X_train['proto'].where(X_train['proto'].isin(keep_protos), 'other')
    X_val['proto'] = X_val['proto'].where(X_val['proto'].isin(keep_protos), 'other')

    X_train = pd.get_dummies(X_train, columns=NOMINAL_COLS).astype(float)
    X_val = pd.get_dummies(X_val, columns=NOMINAL_COLS).reindex(columns=X_train.columns, fill_value=0).astype(float)

    # rank by mutual information with the target and keep the top n
    mi = mutual_info_classif(X_train, train[LABEL], random_state=SEED)
    top = pd.Series(mi, index=X_train.columns).sort_values(ascending=False).head(n_features).index.tolist()
    return X_train[top], X_val[top], top

def smote_tomek_5x(X, y):
    counts = y.value_counts()
    target = {cls: min(n * 5, counts.max()) for cls, n in counts.items()}
    smote = SMOTE(sampling_strategy=target, random_state=SEED, k_neighbors=5)
    return SMOTETomek(smote=smote, random_state=SEED).fit_resample(X, y)

# build the 5 resampled folds once, the k search below reuses them
knn_folds = []
for fold in range(5):
    train, val = dev[dev['fold'] != fold], dev[dev['fold'] == fold]
    X_train, X_val, top = prep_knn_fold(train, val)
    X_res, y_res = smote_tomek_5x(X_train, train[LABEL])
    knn_folds.append((X_res, y_res, X_val, val[LABEL]))
    print(f"fold {fold}: resampled {X_res.shape}, features {top}")

fold 0: resampled (221589, 10), features ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'sload', 'ct_state_ttl', 'dttl', 'dpkts', 'dinpkt']
fold 1: resampled (224692, 10), features ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'sload', 'ct_state_ttl', 'dttl', 'dinpkt', 'dload']
fold 2: resampled (220722, 10), features ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'sload', 'ct_state_ttl', 'dttl', 'dload', 'dpkts']
fold 3: resampled (224642, 10), features ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'sload', 'ct_state_ttl', 'dttl', 'dload', 'dinpkt']
fold 4: resampled (221309, 10), features ['sbytes', 'dbytes', 'smean', 'dmean', 'sttl', 'sload', 'ct_state_ttl', 'dttl', 'dinpkt', 'dpkts']


In [ ]:
# re-checking k on all 5 folds
# in A2 k=7 was picked on a single fold that was then also used for evaluation
# manhattan distance and distance-weighted voting are carried over from A2
# range goes further than A2 (up to 51) since the best value sat near the top of the old range

k_values = [3, 5, 7, 9, 15, 21, 31, 51]

k_results = []
for k in k_values:
    for fold, (X_res, y_res, X_val, y_val) in enumerate(knn_folds):
        model = KNeighborsClassifier(n_neighbors=k, weights='distance', metric='manhattan', n_jobs=-1)
        model.fit(X_res, y_res)
        k_results.append({'k': k, 'fold': fold,
                          'macro_f1': f1_score(y_val, model.predict(X_val), average='macro')})
    print(f"k={k} done")

k_df = pd.DataFrame(k_results)
k_df.to_csv(RESULTS_DIR + 'knn_k.csv', index=False)

k_summary = k_df.groupby('k')['macro_f1'].agg(['mean', 'std']).round(4)
print()
print(k_summary)

k=3 done
k=5 done
k=7 done
k=9 done
k=15 done
k=21 done
k=31 done
k=51 done

      mean     std
k                 
3   0.6065  0.0029
5   0.6121  0.0033
7   0.6164  0.0062
9   0.6163  0.0066
15  0.6209  0.0071
21  0.6220  0.0067
31  0.6240  0.0068
51  0.6219  0.0063


In [ ]:
# final kNN on the development folds
# k=31 has the highest mean, and everything from 15 to 51 is within fold-to-fold variation
# so the result is not sensitive to the exact value in that range

KNN_K = int(k_summary['mean'].idxmax())
print("Chosen k:", KNN_K)

knn_results, knn_preds = [], []
for fold, (X_res, y_res, X_val, y_val) in enumerate(knn_folds):
    model = KNeighborsClassifier(n_neighbors=KNN_K, weights='distance', metric='manhattan', n_jobs=-1)
    start = time.time()
    model.fit(X_res, y_res)
    fit_time = time.time() - start
    start = time.time()
    y_pred = model.predict(X_val)
    predict_time = time.time() - start

    knn_results.append({'model': 'knn', 'fold': fold,
                        'macro_f1': f1_score(y_val, y_pred, average='macro'),
                        'accuracy': accuracy_score(y_val, y_pred),
                        'weighted_f1': f1_score(y_val, y_pred, average='weighted'),
                        'balanced_accuracy': balanced_accuracy_score(y_val, y_pred),
                        'fit_time': fit_time, 'predict_time': predict_time})
    knn_preds.append(pd.DataFrame({'fold': fold, 'y_true': y_val.values, 'y_pred': y_pred}))

knn_results_df = pd.DataFrame(knn_results)
knn_results_df.to_csv(RESULTS_DIR + 'baseline_knn_cv.csv', index=False)
pd.concat(knn_preds).to_csv(RESULTS_DIR + 'baseline_knn_cv_preds.csv', index=False)

print()
print(knn_results_df.round(4))

# both baselines side by side
both = pd.concat([tree_results_df, knn_results_df])
print()
print(both.drop(columns='fold').groupby('model').agg(['mean', 'std']).round(4).T)

Chosen k: 31

  model  fold  macro_f1  accuracy  weighted_f1  balanced_accuracy  fit_time  \
0   knn     0    0.6133    0.7992       0.8088             0.6249    0.6259   
1   knn     1    0.6229    0.8005       0.8103             0.6378    0.6442   
2   knn     2    0.6244    0.8047       0.8133             0.6389    0.6039   
3   knn     3    0.6313    0.8023       0.8117             0.6435    0.8934   
4   knn     4    0.6283    0.8018       0.8105             0.6421    0.5721   

   predict_time  
0        6.1315  
1        4.7666  
2        5.5354  
3        5.6106  
4        4.3148  

model                      knn    tree
macro_f1          mean  0.6240  0.6695
                  std   0.0068  0.0110
accuracy          mean  0.8017  0.8012
                  std   0.0021  0.0031
weighted_f1       mean  0.8109  0.8148
                  std   0.0017  0.0024
balanced_accuracy mean  0.6374  0.7051
                  std   0.0074  0.0136
fit_time          mean  0.6679  9.8450
            